# Alternative Investment Sales Strategy Analytics Workflow

**Author:** Allen Xu

This lightweight notebook documents the reproducible workflow behind the portfolio project. It reads the generated CSV outputs, summarizes the core sales strategy analytics, and points to the executive report and dashboard-ready artifacts.

> Disclosure: all data is synthetic and generated from `src/generate_synthetic_data.py`. No real KKR, client, advisor, fund, or firm data is used.

## 1. Load Project Outputs

Run the full pipeline from the project root before using this notebook:

```bash
python3.11 src/generate_synthetic_data.py
python3.11 src/build_sqlite_database.py
python3.11 src/run_analysis.py
python3.11 src/create_charts.py
```

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CHARTS_DIR = PROJECT_ROOT / "reports" / "charts"

raw_files = {
    "advisors": "advisors.csv",
    "relationship_managers": "relationship_managers.csv",
    "funds": "funds.csv",
    "sales_activities": "sales_activities.csv",
    "opportunities": "opportunities.csv",
    "campaigns": "campaigns.csv",
    "campaign_engagement": "campaign_engagement.csv",
}

processed_files = {
    "executive_kpis": "executive_kpi_summary.csv",
    "sales_funnel": "sales_funnel_summary.csv",
    "rm_productivity": "rm_productivity_summary.csv",
    "product_demand": "product_demand_summary.csv",
    "campaign_roi": "campaign_roi_summary.csv",
    "advisor_priority": "advisor_priority_scores.csv",
}

raw = {name: pd.read_csv(RAW_DIR / file_name) for name, file_name in raw_files.items()}
processed = {name: pd.read_csv(PROCESSED_DIR / file_name) for name, file_name in processed_files.items()}

pd.DataFrame({
    "table": list(raw.keys()),
    "rows": [len(df) for df in raw.values()],
})

## 2. Executive KPI Readout

These KPIs are the same deterministic values used in the README and executive report.

In [ ]:
kpis = processed["executive_kpis"].set_index("kpi")["value"]

summary = pd.DataFrame([
    ["Expected pipeline", f"${kpis['total_pipeline_value_usd'] / 1e9:,.1f}B"],
    ["Committed capital", f"${kpis['committed_capital_usd'] / 1e9:,.2f}B"],
    ["Overall conversion rate", f"{kpis['overall_conversion_rate_pct']:.2f}%"],
    ["Average days to close", f"{kpis['avg_days_to_close']:.0f}"],
    ["Campaign ROI", f"{kpis['portfolio_campaign_roi_pct']:,.0f}%"],
    ["High-priority advisors", f"{int(kpis['high_priority_advisors']):,}"],
    ["Coaching-opportunity RMs", f"{int(kpis['coaching_opportunity_rms']):,}"],
], columns=["metric", "value"])

summary

## 3. Sales Funnel And Product Demand

The funnel view isolates stage drop-off and conversion by region, firm type, asset class, and AUM segment.

In [ ]:
funnel = processed["sales_funnel"]
stage_order = ["Prospect", "Interested", "Due Diligence", "Soft Circle", "Committed", "Lost"]

(funnel[funnel["group_label"] == "stage"]
 .assign(group_value=lambda df: pd.Categorical(df["group_value"], stage_order, ordered=True))
 .sort_values("group_value")
 [["group_value", "total_opportunities", "expected_pipeline_usd", "committed_capital_usd", "committed_rate"]])

In [ ]:
product_demand = processed["product_demand"]

(product_demand[product_demand["dimension"] == "asset_class"]
 .sort_values("committed_capital", ascending=False)
 [["value", "activities", "avg_engagement", "opportunities", "committed_capital", "conversion_rate_pct"]])

## 4. Relationship Manager Productivity

The RM summary flags productivity leaders and coaching opportunities where engagement is strong but conversion lags the team.

In [ ]:
rm_productivity = processed["rm_productivity"]

rm_productivity.sort_values("committed_capital", ascending=False).head(8)[[
    "rm_id", "rm_name", "region", "sales_team", "total_activities",
    "conversion_rate_pct", "committed_capital", "commitment_per_meeting_usd", "coaching_flag"
]]

## 5. Campaign ROI

Campaign ROI uses committed capital, campaign cost, qualified opportunities, and a 2.5% management-fee proxy.

In [ ]:
campaign_roi = processed["campaign_roi"]

(campaign_roi.groupby("campaign_type")
 .agg(campaigns=("campaign_id", "count"),
      spend=("campaign_cost", "sum"),
      qualified_opps=("qualified_opps", "sum"),
      committed_capital=("committed_capital", "sum"))
 .assign(committed_capital_to_cost_multiple=lambda df: df["committed_capital"] / df["spend"])
 .sort_values("committed_capital_to_cost_multiple", ascending=False))

## 6. Advisor Priority Scoring

The priority score ranks advisors for weekly RM coverage planning using engagement, AUM, conversion probability, product fit, and recency.

In [ ]:
advisor_priority = processed["advisor_priority"]

advisor_priority.sort_values("priority_score", ascending=False).head(10)[[
    "advisor_id", "firm_name", "firm_type", "region", "aum_segment",
    "priority_score", "priority_tier", "recommended_next_action",
    "recommended_asset_class", "estimated_commitment_opportunity"
]]

## 7. Reporting Artifacts

- Executive report: `reports/executive_report.md`
- Dashboard build notes: `dashboard/tableau_powerbi_notes.md`
- Dashboard data model: `dashboard/dashboard_data_model.md`
- Dashboard wireframe: `dashboard/dashboard_wireframe.md`
- Generated charts:
  - `reports/charts/sales_funnel_conversion.png`
  - `reports/charts/rm_productivity.png`
  - `reports/charts/campaign_roi.png`
  - `reports/charts/product_demand.png`